# 🏥 Clinical Knowledge RAG — Demo Notebook

**vLLM · FAISS · Anaconda CLI + Outerbounds**

> **Owner:** Federico Rubiano | **Last tested:** 2026-05-18 | **Status:** Active | **Estimated time:** 45–60 minutes

---

> ## ⚠️ MEDICAL DISCLAIMER
>
> **This notebook is for educational and demonstration purposes only.**
>
> Nothing produced by this system — including all generated text, citations, and clinical summaries — constitutes medical advice, diagnosis, or treatment. The system may produce inaccurate, incomplete, or outdated information even when citing real sources.
>
> **Always consult a qualified, licensed healthcare professional before making any clinical decision.** Do not use this tool in any real patient care setting. The authors, contributors, and Anaconda, Inc. accept no liability whatsoever for decisions made on the basis of this system's output.

---

This notebook walks through the full Clinical RAG pipeline end-to-end.
Run it after the environment is set up and indexes are built.

### What you'll see
1. V1 vs V2 — the failure modes we fixed
2. The retrieval pipeline (BM25 + FAISS + RRF + cross-encoder)
3. vLLM inference with citation enforcement
4. Live API calls via FastAPI
5. Evaluation scores from `eval/run_eval.py`
6. vLLM throughput demo — sequential vs. concurrent (PagedAttention)
7. MCP integration with Claude Desktop
8. Full Anaconda CLI deployment flow

---

### Prerequisites
```bash
# 1. Activate the project environment
conda activate vllm-clinical-rag

# 2. Build indexes (if not done yet)
python scripts/scraper.py
python scripts/build_index.py

# 3. Start vLLM (in a separate terminal) — skip if using Outerbounds
bash scripts/start_vllm.sh

# 4. Start the API (in a separate terminal)
uvicorn src.api:app --host 0.0.0.0 --port 8000

# 5. Verify everything is working
python test_api.py
```

## 0. Environment setup

In [ ]:
import sys
import os

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"Python: {sys.version}")

In [ ]:
# Check key packages are installed
import importlib

packages = {
    'faiss': 'faiss-gpu',
    'vllm': 'vllm',
    'sentence_transformers': 'sentence-transformers',
    'rank_bm25': 'rank-bm25',
    'fastapi': 'fastapi',
    'gradio': 'gradio',
    'evidently': 'evidently',
    'openai': 'openai',
}

print("Package status:")
for module, pkg in packages.items():
    try:
        m = importlib.import_module(module)
        version = getattr(m, '__version__', 'installed')
        print(f"  ✅  {pkg:<28} {version}")
    except ImportError:
        print(f"  ❌  {pkg:<28} NOT INSTALLED")

## 1. V1 vs V2 — What changed and why

| Failure mode (V1) | Root cause | V2 fix |
|---|---|---|
| Hallucinated citations | LLM made up sources | Retrieved chunk metadata enforced in response schema |
| Outdated PDF corpus | Watermarked, potentially stale | Live scraper targeting merckmanuals.com |
| Single-threaded inference | llama-cpp; no concurrency | vLLM with PagedAttention + continuous batching |
| Self-judging evaluation | Mistral scored its own output | Evidently AI + heuristic cross-check |
| BM25 only | Missed semantic similarity | FAISS dense + BM25 sparse → RRF merge + cross-encoder rerank |

### Architecture

```
User Query
    │
    ▼
HybridRetriever
    ├── BM25 sparse search  ─┐
    │                        ├── RRF merge → CrossEncoder rerank → top-4 chunks
    └── FAISS dense search  ─┘
    │
    ▼
VLLMClient (Mistral-7B-Instruct via vLLM)
    │   System: citation rules + structure rules
    │   User:   context blocks + question
    ▼
Answer + Citations + Disclaimer
    │
    ├── FastAPI /query  →  Gradio UI
    └── MCP server     →  Claude Desktop
```

## 2. Retrieval pipeline walkthrough

In [ ]:
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))

from src.retriever import HybridRetriever

retriever = HybridRetriever(
    faiss_path=os.path.join(project_root, 'data/index/merck.faiss'),
    bm25_path=os.path.join(project_root, 'data/index/bm25.pkl'),
    chunks_path=os.path.join(project_root, 'data/index/chunks.json'),
)

print(f"Retriever loaded: {len(retriever.chunks)} chunks indexed")

In [ ]:
# Run a demo query through the retrieval pipeline
DEMO_QUERY = "What is the protocol for managing sepsis in a critical care unit?"

print(f"Query: {DEMO_QUERY}")
print("\n--- Retrieving...")

chunks = retriever.retrieve(DEMO_QUERY)

print(f"\nTop {len(chunks)} chunks retrieved:\n")
for i, chunk in enumerate(chunks, 1):
    print(f"[{i}] Section : {chunk.section}")
    print(f"    Score   : {chunk.score:.4f}")
    print(f"    URL     : merckmanuals.com{chunk.url}")
    print(f"    Preview : {chunk.text[:150].strip()}...")
    print()

In [ ]:
# Compare BM25 vs FAISS vs RRF — see the fusion effect
import numpy as np

dense_results  = retriever._dense_search(DEMO_QUERY)
sparse_results = retriever._sparse_search(DEMO_QUERY)
merged         = retriever._rrf_merge(dense_results, sparse_results)

dense_ids  = [idx for idx, _ in dense_results[:5]]
sparse_ids = [idx for idx, _ in sparse_results[:5]]
merged_ids = merged[:5]

print("Top-5 chunk indices comparison:")
print(f"  FAISS dense : {dense_ids}")
print(f"  BM25 sparse : {sparse_ids}")
print(f"  RRF merged  : {merged_ids}")

overlap_d_s = len(set(dense_ids) & set(sparse_ids))
print(f"\n  Dense/Sparse overlap (top 5) : {overlap_d_s}/5")
print(f"  RRF unique contributions     : {len(set(merged_ids) - set(dense_ids[:5]) - set(sparse_ids[:5]))} new chunk(s)")

## 3. vLLM inference — direct client call

> **Requires**: vLLM server running at `http://localhost:8001/v1`  
> Start it with: `bash scripts/start_vllm.sh`

In [ ]:
from src.vllm_client import VLLMClient

llm = VLLMClient()  # reads VLLM_BASE_URL and VLLM_MODEL from env

print(f"vLLM endpoint : {llm.base_url}")
print(f"Model         : {llm.model}")

In [ ]:
# Generate a grounded clinical answer
result = llm.generate(
    question=DEMO_QUERY,
    chunks=chunks,
    max_tokens=512,
    temperature=0.1,
)

print("=" * 70)
print("ANSWER")
print("=" * 70)
print(result['answer'])
print()
print(f"Tokens used : {result['usage']}")

## 4. FastAPI — live /query endpoint

> **Requires**: API server running at `http://localhost:8000`  
> Start it with: `uvicorn src.api:app --host 0.0.0.0 --port 8000`

In [ ]:
import requests

API_URL = os.getenv('API_URL', 'http://localhost:8000')

# Health check
health = requests.get(f'{API_URL}/health').json()
print("Health check:")
for k, v in health.items():
    print(f"  {k:<18} {v}")

In [ ]:
# Run all 5 benchmark queries through the API
BENCHMARK_QUERIES = [
    "What is the protocol for managing sepsis in a critical care unit?",
    "What are the common symptoms for appendicitis, and can it be cured via medicine?",
    "What are the effective treatments for sudden patchy hair loss on the scalp?",
    "What treatments are recommended for traumatic brain injury?",
    "What are the precautions and treatment steps for a leg fracture during a hiking trip?",
]

import time

for i, q in enumerate(BENCHMARK_QUERIES, 1):
    t0 = time.perf_counter()
    resp = requests.post(f'{API_URL}/query', json={'question': q, 'max_tokens': 512}).json()
    elapsed = int((time.perf_counter() - t0) * 1000)
    
    answer_preview = resp.get('answer', '')[:120].replace('\n', ' ')
    sources = [s['section'] for s in resp.get('sources', [])]
    
    print(f"[{i}] Q: {q[:60]}...")
    print(f"     Latency : {resp.get('latency_ms', elapsed)} ms")
    print(f"     Sources : {sources}")
    print(f"     Answer  : {answer_preview}...")
    print()

## 5. Evaluation with Evidently AI

In [ ]:
# Run the evaluation harness
# This calls eval/run_eval.py programmatically

import subprocess

result = subprocess.run(
    [
        sys.executable, 
        os.path.join(project_root, 'eval', 'run_eval.py'),
        '--api-url', API_URL,
        '--output', os.path.join(project_root, 'eval', 'results.json'),
        '--report',
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
# Load and display results
import json
import pandas as pd

results_path = os.path.join(project_root, 'eval', 'results.json')

if os.path.exists(results_path):
    with open(results_path) as f:
        eval_data = json.load(f)

    print(f"Evaluation run: {eval_data['eval_timestamp']}")
    print(f"Model: {eval_data['model']}")
    print()

    # Aggregate scores
    agg = eval_data['aggregate']
    print("Aggregate scores:")
    agg_df = pd.DataFrame([{
        'Metric': k,
        'Score': v
    } for k, v in agg.items() if k not in ('queries_scored', 'queries_failed', 'avg_latency_ms')])
    display(agg_df.set_index('Metric'))

    # Per-query scores
    print("\nPer-query scores:")
    rows = []
    for r in eval_data['results']:
        row = {'query_id': r['id']}
        row.update(r.get('scores', {}))
        row['latency_ms'] = r.get('latency_ms', 0)
        rows.append(row)
    display(pd.DataFrame(rows).set_index('query_id').round(3))
else:
    print(f"No results file found at {results_path}. Run the API and evaluation first.")

## 6. vLLM throughput — PagedAttention demo

The key V2 advantage: concurrent requests. V1 (llama-cpp) processed one query at a time.
vLLM's PagedAttention allows continuous batching — multiple requests share GPU memory efficiently.

In [ ]:
import concurrent.futures
import time

def time_query(q):
    t0 = time.perf_counter()
    resp = requests.post(f'{API_URL}/query', json={'question': q, 'max_tokens': 256}, timeout=120)
    latency = int((time.perf_counter() - t0) * 1000)
    return q[:40], latency

queries = BENCHMARK_QUERIES[:3]

# Sequential baseline
print("Sequential (one at a time):")
t_seq_start = time.perf_counter()
for q in queries:
    label, lat = time_query(q)
    print(f"  {label}... → {lat} ms")
t_seq = int((time.perf_counter() - t_seq_start) * 1000)
print(f"  Total: {t_seq} ms")

# Concurrent (vLLM batching)
print("\nConcurrent (vLLM PagedAttention):")
t_par_start = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(time_query, q): q for q in queries}
    for future in concurrent.futures.as_completed(futures):
        label, lat = future.result()
        print(f"  {label}... → {lat} ms")
t_par = int((time.perf_counter() - t_par_start) * 1000)
print(f"  Total: {t_par} ms")

if t_seq > 0:
    speedup = round(t_seq / t_par, 2)
    print(f"\n  Speedup with vLLM concurrent batching: {speedup}x")

## 7. MCP — Claude Desktop integration

With `ana mcp setup`, the Clinical Knowledge API is available directly in Claude Desktop
as the `query_merck_manual` tool. Claude calls it automatically when you ask clinical questions.

**Config snippet for `~/Library/Application Support/Claude/claude_desktop_config.json`:**

```json
{
  "mcpServers": {
    "merck-manual-rag": {
      "command": "/path/to/envs/vllm-clinical-rag/bin/python",
      "args": ["-m", "src.mcp_server"],
      "cwd": "/path/to/vllm-clinical-rag",
      "env": { "PYTHONPATH": "/path/to/vllm-clinical-rag" }
    }
  }
}
```

Or use the Anaconda CLI helper:

```bash
ana mcp setup
```

Then in Claude Desktop, ask: *"What is the first-line treatment for septic shock?"*  
Claude will query the Merck Manual RAG instead of using training data.

## 8. Anaconda CLI — full deployment flow

```bash
# Step 1: Login
ana login

# Step 2: Enable main-x (early access packages: vLLM, FAISS, Gradio)
ana feature enable main-x

# Step 3: Create environment from project spec
conda env create -f environment.yml
conda activate vllm-clinical-rag

# Step 4: Build data indexes
python scripts/scraper.py
python scripts/build_index.py

# Step 5: Start vLLM + API locally
bash scripts/start_vllm.sh           # terminal 1
uvicorn src.api:app --port 8000       # terminal 2

# Step 6: (Optional) Launch Gradio demo UI
python src/gradio_app.py              # terminal 3

# Step 7: (Optional) Connect to Claude Desktop
ana mcp setup

# Step 8: Run evaluation
python eval/run_eval.py --report

# Step 9: Deploy to Outerbounds
ana ob deploy
```

This is the **first install to production** story:
- `ana login` → authenticated
- `ana feature enable main-x` → latest AI packages unlocked
- `ana ob deploy` → production inference serving at scale

In [ ]:
print("Demo complete ✅")
print()
print("High-value packages showcased in this demo:")
packages_used = [
    ("vLLM",                 "Production LLM inference with PagedAttention"),
    ("FAISS",                "GPU-accelerated vector similarity search"),
    ("Gradio",               "Interactive demo UI"),
    ("Evidently AI",         "RAG evaluation and monitoring"),
    ("sentence-transformers","Embedding + cross-encoder reranking"),
    ("FastAPI",              "Production REST API"),
    ("rank-bm25",            "Sparse BM25 keyword search"),
]
for pkg, desc in packages_used:
    print(f"  • {pkg:<26} {desc}")